### NEW - SM Chile, France & Brazil

In [19]:
import svgutils.transform as st
from svgutils.compose import Text
import cairosvg

# -----------------------------
# Layout settings (3 panels)
# -----------------------------

H_GAP = -220
TARGET_WIDTH = 500          # BIGGER panels
TARGET_HEIGHT = 500
H_GAP = 1                  # SMALLER gap between panels
LEFT_MARGIN = 1
RIGHT_MARGIN = 1

TOP_MARGIN = 1
TITLE_MARGIN = 42          # space above each plot for title
BOTTOM_MARGIN = 1

TITLE_FONT_SIZE = 18
TITLE_WEIGHT = "bold"

# -----------------------------
# Input SVGs
# -----------------------------
files = [
    ("Output/SM-heatmap/heatmap_chile.svg",  "a) Chile"),
    ("Output/SM-heatmap/heatmap_france.svg", "b) France"),
    ("Output/SM-heatmap/heatmap_brazil.svg", "c) Brazil"),
]

# -----------------------------
# Helpers
# -----------------------------
def safe_get_size(fig, default_w=800, default_h=600):
    try:
        w, h = fig.get_size()
        w = float(str(w).replace("px", "")) if w else default_w
        h = float(str(h).replace("px", "")) if h else default_h
        return w, h
    except Exception:
        return default_w, default_h

def scale_to_fit(element, orig_w, orig_h, target_w, target_h):
    scale = min(target_w / orig_w, target_h / orig_h)
    element.scale(scale)
    return orig_w * scale, orig_h * scale

# -----------------------------
# Build combined figure
# -----------------------------
elements = []
max_content_height = 0

x_positions = [
    LEFT_MARGIN + i * (TARGET_WIDTH + H_GAP)
    for i in range(len(files))
]

for i, (filepath, title) in enumerate(files):
    x = x_positions[i]
    y = TOP_MARGIN

    fig = st.fromfile(filepath)
    root = fig.getroot()

    # Remove any inherited transforms
    try:
        root.root.attrib.pop("transform", None)
    except Exception:
        pass

    w, h = safe_get_size(fig)
    scaled_w, scaled_h = scale_to_fit(root, w, h, TARGET_WIDTH, TARGET_HEIGHT)

    # Move plot (leave space for title)
    root.moveto(x, y + TITLE_MARGIN)

    title_text = Text(
        title,
        x + 5,
        y + int(TITLE_MARGIN * 0.75),
        size=TITLE_FONT_SIZE,
        weight=TITLE_WEIGHT,
    )

    elements.append(title_text)
    elements.append(root)

    max_content_height = max(max_content_height, y + TITLE_MARGIN + scaled_h)

# -----------------------------
# Final canvas
# -----------------------------
final_width = int(
    LEFT_MARGIN
    + len(files) * TARGET_WIDTH
    + (len(files) - 1) * H_GAP
    + RIGHT_MARGIN
)

final_height = int(max_content_height + BOTTOM_MARGIN)

combined_fig = st.SVGFigure(f"{final_width}px", f"{final_height}px")
combined_fig.root.attrib["viewBox"] = f"0 0 {final_width} {final_height}"
combined_fig.append(elements)

plt.savefig(path, format="svg", bbox_inches="tight", pad_inches=0.01)


# -----------------------------
# Save SVG + PDF
# -----------------------------
svg_path = "Output/SM-heatmap/combined_heatmaps.svg"
pdf_path = "Output/SM-heatmap/combined_heatmaps.pdf"

combined_fig.save(svg_path)
cairosvg.svg2pdf(url=svg_path, write_to=pdf_path)

print(f"✅ Combined SVG saved: {svg_path}")
print(f"✅ Combined PDF saved: {pdf_path}")
print(f"Canvas: {final_width}px x {final_height}px")


✅ Combined SVG saved: Output/SM-heatmap/combined_heatmaps.svg
✅ Combined PDF saved: Output/SM-heatmap/combined_heatmaps.pdf
Canvas: 1504px x 419px
